<a href="https://colab.research.google.com/github/brunodino/workshop_dados/blob/main/notebooks/dia1_fundamentos_cleaning_REV00.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop de Análise de Dados - Dia 1
## Fundamentos e Data Cleaning

Hoje você vai receber um dataset real de um marketplace brasileiro e fazer o que todo analista faz no primeiro contato com dados novos: entender o que tem ali, limpar o que precisa ser limpo, explorar o que chama atenção, e formular as perguntas certas.

Ao final do dia, você terá:
- Um dataset limpo e documentado
- Suas primeiras visualizações exploratórias
- Perguntas de negócio definidas e priorizadas para o Dia 2

## Briefing

Leia o briefing abaixo. Ele simula o que você receberia de um gestor no mundo real. As perguntas são abertas de propósito.

> **De:** Marina Silva, Gerente de Operações  
> **Assunto:** Precisamos entender melhor o que está acontecendo com a satisfação dos clientes
>
> 1. **Satisfação por categoria:** Existem categorias onde a insatisfação é sistematicamente pior? Qual o tamanho de cada segmento?
> 2. **Relação entre frete e satisfação:** Pedidos com frete caro ou entrega atrasada geram mais avaliações negativas?
> 3. **Sazonalidade e tendência:** O volume tem sazonalidade? As notas pioram em picos?
>
> Preciso de diagnóstico visual para a diretoria. Se encontrarem inconsistências, documentem.

---
# Bloco 1: Diagnóstico do Dataset (45 min)

**Objetivo:** Conhecer os dados antes de tocar em qualquer coisa. Entender estrutura, volume, tipos, e identificar problemas.

In [1]:
# Setup - execute esta célula sem alterar
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas numpy matplotlib seaborn
    !mkdir -p data/prepared
    # Baixar datasets preparados do workshop
    BASE_URL = 'https://github.com/caiocgomes/workshop_dados/releases/download/v1/'
    for f in ['orders.csv', 'items.csv', 'reviews.csv']:
        !wget -q -O data/prepared/{f} {BASE_URL}/{f}
    DATA_DIR = 'data/prepared'
    print('Ambiente Colab configurado!')
else:
    DATA_DIR = '../data/prepared'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

print('Setup completo!')

Ambiente Colab configurado!
Setup completo!


In [2]:
# Carregue os 3 datasets
orders = pd.read_csv(f'{DATA_DIR}/orders.csv')
items = pd.read_csv(f'{DATA_DIR}/items.csv')
reviews = pd.read_csv(f'{DATA_DIR}/reviews.csv')

print(f'Orders:  {orders.shape[0]:,} linhas, {orders.shape[1]} colunas')
print(f'Items:   {items.shape[0]:,} linhas, {items.shape[1]} colunas')
print(f'Reviews: {reviews.shape[0]:,} linhas, {reviews.shape[1]} colunas')

Orders:  99,941 linhas, 7 colunas
Items:   112,650 linhas, 6 colunas
Reviews: 99,224 linhas, 6 colunas


### 1.1 Visão geral das tabelas

Antes de qualquer análise: olhe os tipos, as primeiras linhas, e os nomes das colunas. Execute as duas células abaixo para ter uma visão rápida das 3 tabelas.

In [3]:
# Veja as primeiras linhas de cada tabela
print('=== ORDERS ===')
display(orders.head())
print('\n=== ITEMS ===')
display(items.head())
print('\n=== REVIEWS ===')
display(reviews.head())

=== ORDERS ===


,order_id,customer_id,order_status,purchase_date,approved_date,delivered_date,estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,08/08/2018 08:38:49,2018-08-08 08:55:23,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-16 18:17:02,2018-02-26 00:00:00



=== ITEMS ===


,order_id,order_item_id,product_id,category,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,pet_shop,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,perfumery,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,garden_tools,199.90,18.14



=== REVIEWS ===


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,NaN,NaN,NaN,2018-01-18 00:00:00
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5.0,NaN,NaN,2018-03-10 00:00:00
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5.0,NaN,NaN,2018-02-17 00:00:00
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5.0,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,NaN,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00


In [4]:
# Verifique os tipos de cada coluna
# Pergunta para você: algum tipo parece errado? Datas como string? Números como object?
print('=== ORDERS dtypes ===')
print(orders.dtypes)
print('\n=== ITEMS dtypes ===')
print(items.dtypes)
print('\n=== REVIEWS dtypes ===')
print(reviews.dtypes)

=== ORDERS dtypes ===
order_id                   object
customer_id                object
order_status               object
purchase_date              object
approved_date              object
delivered_date             object
estimated_delivery_date    object
dtype: object

=== ITEMS dtypes ===
order_id          object
order_item_id      int64
product_id        object
category          object
price            float64
freight_value    float64
dtype: object

=== REVIEWS dtypes ===
review_id                  object
order_id                   object
review_score              float64
review_comment_title       object
review_comment_message     object
review_creation_date       object
dtype: object


### 1.2 Demo: investigando `orders` (o instrutor faz ao vivo)

O instrutor vai demonstrar o processo de diagnóstico usando a tabela `orders`. Acompanhe a sequência: missings, duplicatas, estatísticas descritivas. Depois, você faz o mesmo para `items` e `reviews`.

In [5]:
# Demo: missings em orders (o instrutor verbaliza o raciocínio)
missing_orders = orders.isnull().sum()
missing_orders = missing_orders[missing_orders > 0]
if len(missing_orders) > 0:
    print('=== ORDERS - Valores ausentes ===')
    print(missing_orders)
else:
    print('=== ORDERS - Sem valores ausentes ===')

# Resumo rápido de missings nas 3 tabelas (pra saber onde investigar)
print('\n--- Resumo de missings nas 3 tabelas ---')
for name, df in [('orders', orders), ('items', items), ('reviews', reviews)]:
    total_missing = df.isnull().sum().sum()
    print(f'  {name}: {total_missing} valores ausentes')

=== ORDERS - Valores ausentes ===
approved_date      161
delivered_date    2978
dtype: int64

--- Resumo de missings nas 3 tabelas ---
  orders: 3139 valores ausentes
  items: 1627 valores ausentes
  reviews: 151793 valores ausentes


### 1.2b Duplicatas em `orders` (continuação da demo)

In [ ]:
# Verifique duplicatas em orders
print(f'Duplicatas exatas em orders: {orders.duplicated().sum()}')
print(f'Order_ids duplicados: {orders["order_id"].duplicated().sum()}')

# Se order_id tem duplicatas mas as linhas não são idênticas, investigue:
dup_ids = orders[orders['order_id'].duplicated(keep=False)]['order_id'].unique()
print(f'\nPedidos com mais de um registro: {len(dup_ids)}')

# Veja alguns exemplos
if len(dup_ids) > 0:
    sample_id = dup_ids[0]
    print(f'\nExemplo - pedido {sample_id}:')
    display(orders[orders['order_id'] == sample_id])

---

### 1.3 Seu turno: investigue `items`

Agora é com você. Siga o mesmo processo que o instrutor fez com `orders`: olhe as estatísticas descritivas, procure valores estranhos, investigue colunas categóricas. Complete o código nas células abaixo.

In [ ]:
# Estatísticas descritivas de items
# Execute e olhe com atenção: o min e max de price fazem sentido para um marketplace?
items_describe = items.describe()
display(items_describe)

# Escreva aqui o que chamou sua atenção no describe:
#

In [ ]:
# Investigue as categorias de produto
# Quantas categorias únicas existem? Alguma parece duplicada com grafia diferente?

# COMPLETE: conte os valores únicos de 'category'
n_categorias = ___
print(f'Total de categorias únicas: {n_categorias}')

# COMPLETE: mostre as 20 categorias mais frequentes
top_categorias = ___
print(f'\nTop 20 categorias:\n{top_categorias}')

# COMPLETE: mostre as categorias MENOS frequentes (com menos de 50 itens)
# Dica: use value_counts() e filtre
cats = items['category'].value_counts()
categorias_raras = ___
print(f'\nCategorias com poucos itens (possíveis typos?):\n{categorias_raras}')

# O que você observa? Tem categorias que parecem ser a mesma coisa com nomes diferentes?
# Sua observação:

---

### 1.4 Seu turno: investigue `reviews`

Mesmo processo. Aqui o desafio é maior: além de olhar o que está nos dados, investigue o que **falta** — os missings. Missings nem sempre são aleatórios.

In [ ]:
# Estatísticas descritivas de reviews
reviews_describe = reviews.describe()
display(reviews_describe)

# Quantos missings tem em review_score?
n_missing_reviews = reviews['review_score'].isna().sum()
pct_missing_reviews = n_missing_reviews / len(reviews) * 100
print(f'\nMissings em review_score: {n_missing_reviews} ({pct_missing_reviews:.1f}%)')

# 6% parece pouco. Mas será que esses 6% são aleatórios?
# Para investigar, junte reviews com items e compare o preço médio
# dos pedidos COM e SEM review_score

# COMPLETE: faça o merge de reviews com items pela coluna 'order_id'
reviews_items = ___

# COMPLETE: calcule o preço médio dos pedidos COM review_score
preco_com_score = ___
print(f'\nPreço médio - COM review_score: R${preco_com_score:.2f}')

# COMPLETE: calcule o preço médio dos pedidos SEM review_score
preco_sem_score = ___
print(f'Preço médio - SEM review_score: R${preco_sem_score:.2f}')

# O que você observa? Os missings parecem aleatórios?
# Sua observação:

**Documente aqui os problemas que encontrou no diagnóstico:**

1.
2.
3.
4.
5.

### Checkpoint - Bloco 1

Execute a célula abaixo para verificar se você completou o diagnóstico.

In [ ]:
# CHECKPOINT 1 - Diagnóstico
checks = {
    '3 DataFrames carregados': all(v in dir() for v in ['orders', 'items', 'reviews']),
    'Investigou items (describe)': 'items_describe' in dir(),
    'Investigou categorias': 'n_categorias' in dir(),
    'Investigou reviews (describe)': 'reviews_describe' in dir(),
    'Investigou missings de reviews': 'reviews_items' in dir(),
}
for check, passed in checks.items():
    status = 'OK' if passed else 'FALTA'
    print(f'  [{status}] {check}')

n_ok = sum(checks.values())
if n_ok == len(checks):
    print('\nBloco 1 completo! Siga para o Bloco 2.')
elif n_ok >= 3:
    print(f'\n{n_ok}/{len(checks)} feitos. Complete os que faltam antes de seguir.')
else:
    print(f'\n{n_ok}/{len(checks)} feitos. Volte e complete os exercícios das seções 1.3 e 1.4.')

### Desafio - Bloco 1 (opcional)

Para quem terminou antes: investigue se os missings em `review_score` estão correlacionados com a **categoria** do produto, além do preço. Use um groupby ou crosstab.

In [ ]:
# Desafio: sua análise aqui


---
# Bloco 2A: Limpeza Estrutural (30 min)

**Objetivo:** Resolver problemas de integridade dos dados: registros duplicados e formatos inconsistentes. São problemas de **estrutura**, onde o dado não está no formato correto para análise.

Limpeza sem justificativa não é limpeza, é destruição de dados.

### 2.1 Tratamento de duplicatas

As duplicatas em orders não são idênticas: têm timestamps diferentes. Você precisa decidir qual manter.

In [ ]:
# Estratégia: para pedidos com mais de um registro, manter o mais recente (último timestamp)
# Justificativa: o registro mais recente provavelmente reflete o estado final do pedido

print(f'Orders antes: {len(orders)}')

# Ordene por order_id e purchase_date, mantenha o último
orders_clean = orders.sort_values('purchase_date').drop_duplicates(
    subset='order_id',
    keep='last'  # Altere aqui se quiser testar 'first'
)

print(f'Orders depois: {len(orders_clean)}')
print(f'Registros removidos: {len(orders) - len(orders_clean)}')

# Sua justificativa para a escolha:
#

### 2.2 Correção de formatos de data

O campo `purchase_date` mistura formatos (ISO e brasileiro). Precisamos padronizar antes de usar.

In [ ]:
# Veja exemplos dos dois formatos
print('Exemplos de purchase_date:')
print(orders_clean['purchase_date'].sample(10, random_state=42).values)

In [ ]:
# Converta purchase_date para datetime
# pd.to_datetime com format='mixed' tenta inferir o formato de cada valor
date_cols = ['purchase_date', 'approved_date', 'delivered_date', 'estimated_delivery_date']

for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], format='mixed', dayfirst=True)
    print(f'{col}: {orders_clean[col].dtype}, nulls: {orders_clean[col].isna().sum()}')

# Verifique se o range de datas faz sentido
print(f'\nPeriodo dos pedidos: {orders_clean["purchase_date"].min()} a {orders_clean["purchase_date"].max()}')

### Mini-checkpoint: Limpeza Estrutural

Execute a célula abaixo para verificar que duplicatas e datas estão resolvidas antes de seguir.

In [ ]:
# MINI-CHECKPOINT - Limpeza Estrutural
checks = {
    'Duplicatas removidas': orders_clean['order_id'].duplicated().sum() == 0,
    'Datas convertidas': orders_clean['purchase_date'].dtype == 'datetime64[ns]',
    'Orders com ~99k linhas': 98000 < len(orders_clean) < 100000,
}
for check, passed in checks.items():
    status = 'OK' if passed else 'VERIFICAR'
    print(f'  [{status}] {check}')

if all(checks.values()):
    print('\nLimpeza estrutural OK! Siga para a limpeza semântica.')
else:
    print('\nRevise os itens acima antes de continuar.')

---
# Bloco 2B: Limpeza Semântica (35 min)

**Objetivo:** Resolver problemas de significado dos dados: valores ausentes não-aleatórios, categorias com erros de grafia, e outliers que podem distorcer a análise. São problemas onde o dado existe mas pode enganar.

### 2.3 Tratamento de missings

Já sabemos que os missings em `review_score` não são aleatórios. Precisamos decidir o que fazer com eles.

In [ ]:
# Opções para missings em review_score:
# a) Dropar linhas com missing (perde ~6% dos dados)
# b) Imputar pela mediana
# c) Manter como NaN e filtrar nas análises que usam score
#
# Qual você escolhe e por quê? Descomente UMA opção abaixo.

# Opção A - dropar
# reviews_clean = reviews.dropna(subset=['review_score'])

# Opção B - imputar mediana
# reviews_clean = reviews.copy()
# reviews_clean['review_score'] = reviews_clean['review_score'].fillna(reviews_clean['review_score'].median())

# Opção C - manter NaN
# reviews_clean = reviews.copy()

# === DESCOMENTE SUA ESCOLHA ACIMA E EXECUTE ===
# Se nenhuma opção for descomentada, esta célula vai dar erro de propósito.
print(f'Reviews: {len(reviews_clean)} linhas')
print(f'Review scores válidos: {reviews_clean["review_score"].notna().sum()}')
print(f'Review scores missing: {reviews_clean["review_score"].isna().sum()}')

# Justificativa da sua escolha:
#

### 2.4 Padronização de categorias

O dicionário de correção já está montado. No mundo real, você construiria isso investigando os value_counts como fizemos no diagnóstico.

In [ ]:
# Padronize categorias: lowercase, underscores, sem espaços extras
items_clean = items.copy()
items_clean['category'] = (
    items_clean['category']
    .str.lower()
    .str.strip()
    .str.replace(' ', '_', regex=False)
)

# Veja os typos que ainda restam (categorias com menos de 50 itens):
cats = items_clean['category'].value_counts()
print('Categorias com poucos itens (possíveis typos):')
print(cats[cats < 50])

# Dicionário de correção (já montado — no mundo real, você
# construiria isso investigando os dados)
typo_fixes = {
    'heath_beauty': 'health_beauty',
    'health_beuty': 'health_beauty',
    'sport_leisure': 'sports_leisure',
    'house_wares': 'housewares',
    'housware': 'housewares',
    'computer_accessories': 'computers_accessories',
    'furnitur_decor': 'furniture_decor',
    'watches_gift': 'watches_gifts',
    'telefony': 'telephony',
    'telephoni': 'telephony',
    'bed_bath': 'bed_bath_table',
}

items_clean['category'] = items_clean['category'].replace(typo_fixes)
print(f'\nAntes: {items["category"].nunique()} categorias')
print(f'Depois: {items_clean["category"].nunique()} categorias')

# Por que isso importa? Sem padronizar, 'health_beauty' aparece como
# 3 categorias separadas. Qualquer contagem, média ou ranking fica errado.

### 2.5 Tratamento de outliers

In [ ]:
# Visualize a distribuição de preços
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(items_clean['price'], bins=100)
axes[0].set_title('Distribuição de Preços (todos)')
axes[0].set_xlabel('Preço (R$)')

axes[1].hist(items_clean[items_clean['price'] < 500]['price'], bins=100)
axes[1].set_title('Distribuição de Preços (< R$500)')
axes[1].set_xlabel('Preço (R$)')

plt.tight_layout()
plt.show()

# Percentis extremos
print('Percentis de preço:')
for p in [50, 90, 95, 99, 99.5, 100]:
    print(f'  P{p}: R${items_clean["price"].quantile(p/100):,.2f}')

In [ ]:
# Decisão sobre outliers:
# Os preços extremos não são erros - são compras de volume diferente (possivelmente B2B)
#
# Opções:
# a) Remover itens acima de um threshold
# b) Criar flag 'is_outlier' e manter no dataset
# c) Segmentar análise (B2C vs possível B2B)

# Opção B - criar flag (recomendada: preserva dados, permite análise separada)
threshold = items_clean['price'].quantile(0.99)  # Altere se quiser
items_clean['is_high_value'] = items_clean['price'] > threshold

print(f'Threshold (P99): R${threshold:,.2f}')
print(f'Itens normais: {(~items_clean["is_high_value"]).sum():,}')
print(f'Itens alto valor: {items_clean["is_high_value"].sum():,}')

# Sua justificativa:
#

### Checkpoint - Bloco 2B (Final da Limpeza)

In [ ]:
# CHECKPOINT 2 - Limpeza
checks = {
    'Duplicatas removidas de orders': orders_clean['order_id'].duplicated().sum() == 0,
    'Datas convertidas para datetime': orders_clean['purchase_date'].dtype == 'datetime64[ns]',
    'Orders limpo tem ~99k linhas': 98000 < len(orders_clean) < 100000,
    'Categorias padronizadas': items_clean['category'].nunique() < 80,
    'Flag de outlier criada': 'is_high_value' in items_clean.columns,
}
for check, passed in checks.items():
    status = 'OK' if passed else 'VERIFICAR'
    print(f'  [{status}] {check}')

if all(checks.values()):
    print('\nBloco 2 completo! Siga para o Bloco 3.')
else:
    print('\nAlguns itens precisam de atenção.')

### Desafio - Bloco 2 (opcional)

Crie uma coluna `delivery_delay_days` em `orders_clean` que calcule a diferença entre `delivered_date` e `estimated_delivery_date`. Valores positivos = atraso, negativos = entrega antecipada. Qual a mediana?

In [ ]:
# Desafio: sua análise aqui


---
# Bloco 3: Análise Exploratória Inicial (30 min)

**Objetivo:** Visualizar as distribuições principais e identificar padrões que direcionem as perguntas de negócio. Aqui você está gerando intuição, não respondendo perguntas ainda.

### 3.1 Distribuições univariadas

In [ ]:
# Distribuição de review_score (excluindo NaN)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

reviews_clean['review_score'].dropna().value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue'
)
axes[0].set_title('Distribuição de Notas de Avaliação')
axes[0].set_xlabel('Nota')
axes[0].set_ylabel('Quantidade')

# Distribuição de preços (sem outliers extremos)
items_clean[~items_clean['is_high_value']]['price'].hist(
    bins=50, ax=axes[1], color='coral'
)
axes[1].set_title('Distribuição de Preços (excl. alto valor)')
axes[1].set_xlabel('Preço (R$)')

plt.tight_layout()
plt.show()

In [ ]:
# Volume de pedidos ao longo do tempo
orders_clean.set_index('purchase_date').resample('ME')['order_id'].count().plot(
    figsize=(14, 5), marker='o', color='steelblue'
)
plt.title('Volume de Pedidos por Mês')
plt.xlabel('Mês')
plt.ylabel('Quantidade de Pedidos')
plt.tight_layout()
plt.show()

### 3.2 Exploração por categoria

Quais categorias vendem mais? Quais geram mais receita? São as mesmas?

In [ ]:
# Top 15 categorias por volume e por receita
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Por volume
top_volume = items_clean['category'].value_counts().head(15)
top_volume.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Categorias por Volume')
axes[0].set_xlabel('Quantidade de Itens')

# Por receita
top_receita = items_clean.groupby('category')['price'].sum().nlargest(15)
top_receita.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 15 Categorias por Receita')
axes[1].set_xlabel('Receita Total (R$)')

plt.tight_layout()
plt.show()

# As categorias que vendem mais são as que geram mais receita?
# Sua observação:

### 3.3 Satisfação por status do pedido

O status do pedido (entregue, cancelado, em trânsito) afeta a avaliação? Esse cruzamento já dá uma primeira pista sobre o que importa para o cliente.

In [ ]:
# Boxplot de review_score por status do pedido
orders_reviews = orders_clean.merge(reviews_clean, on='order_id', how='inner')

plt.figure(figsize=(12, 5))
orders_reviews.boxplot(column='review_score', by='order_status', figsize=(12, 5))
plt.title('Review Score por Status do Pedido')
plt.suptitle('')  # Remove título automático do pandas
plt.xlabel('Status')
plt.ylabel('Nota')
plt.show()

# O que a distribuição de notas por status revela?
# Sua observação:

**Documente aqui 2 observações da exploração:**

1.
2.

_Bônus (se tiver tempo):_ 3.

### Checkpoint - Bloco 3

In [ ]:
# CHECKPOINT 3 - Exploração
checks = {
    'Merge orders+reviews existe': 'orders_reviews' in dir(),
    'Merge tem dados': len(orders_reviews) > 0 if 'orders_reviews' in dir() else False,
    'Merge tem colunas esperadas': all(c in orders_reviews.columns for c in ['review_score', 'order_status']) if 'orders_reviews' in dir() else False,
    'Top categorias calculado': 'top_volume' in dir() and 'top_receita' in dir(),
}
for check, passed in checks.items():
    status = 'OK' if passed else 'VERIFICAR'
    print(f'  [{status}] {check}')

print('\nVerifique manualmente:')
print('  [ ] Pelo menos 2 observações documentadas?')
print('Se sim, siga para o Bloco 4!')

### Desafio - Bloco 3 (opcional)

Calcule o tempo médio de entrega por categoria (top 10) e visualize. Há categorias onde a entrega é sistematicamente mais lenta?

In [ ]:
# Desafio: sua análise aqui


---
# Bloco 4: Perguntas Analíticas (55 min)

**Objetivo:** Transformar as observações da exploração em perguntas de negócio concretas e priorizadas. Estas perguntas guiam toda a análise do Dia 2.

### 4.1 Formulação de perguntas

Uma boa pergunta analítica tem 3 características:
1. **Respondível** com os dados disponíveis
2. **Acionável** para o negócio (a resposta muda uma decisão)
3. **Específica** o suficiente para virar uma análise (não vaga)

Releia o briefing da Marina e suas observações. Formule pelo menos 3 perguntas.

### Suas Perguntas Analíticas

**Pergunta 1:**  
_[Escreva aqui]_

- Dados necessários:
- Métrica principal:
- Impacto para o negócio:

**Pergunta 2:**  
_[Escreva aqui]_

- Dados necessários:
- Métrica principal:
- Impacto para o negócio:

**Pergunta 3:**  
_[Escreva aqui]_

- Dados necessários:
- Métrica principal:
- Impacto para o negócio:

### 4.2 Priorização

Não dá para responder tudo. Priorize pelo cruzamento de **impacto** (o quanto muda a decisão) e **viabilidade** (o quanto os dados permitem responder bem).

### Revisão entre pares

Troque suas perguntas com um colega ao lado. Para cada pergunta do colega, avalie usando os 3 critérios:

1. **Respondível?** Os dados que temos permitem responder?
2. **Acionável?** A resposta muda alguma decisão da Marina?
3. **Específica?** Dá para visualizar qual tabela/coluna usar?

Se alguma pergunta falhar em algum critério, sugira como reformulá-la. Depois, volte às suas próprias perguntas e refine com base no feedback.

| Pergunta | Impacto (1-5) | Viabilidade (1-5) | Prioridade |
|----------|:---:|:---:|:---:|
| Pergunta 1 |  |  |  |
| Pergunta 2 |  |  |  |
| Pergunta 3 |  |  |  |

### 4.3 Salvamento dos dados limpos

Execute a célula abaixo **durante a revisão entre pares** — deixe rodar em paralelo enquanto revisa as perguntas do colega. No Colab, os arquivos serão baixados automaticamente; guarde-os para usar no Dia 2.

In [ ]:
# Salve os dados limpos
orders_clean.to_csv(f'{DATA_DIR}/orders_clean.csv', index=False)
items_clean.to_csv(f'{DATA_DIR}/items_clean.csv', index=False)
reviews_clean.to_csv(f'{DATA_DIR}/reviews_clean.csv', index=False)

print('Dados limpos salvos!')
print(f'  orders_clean.csv:  {len(orders_clean):,} linhas')
print(f'  items_clean.csv:   {len(items_clean):,} linhas')
print(f'  reviews_clean.csv: {len(reviews_clean):,} linhas')

# Se estiver no Colab, baixe os arquivos para não perder entre sessões
if IN_COLAB:
    from google.colab import files
    print('\nBaixando arquivos limpos para seu computador (guarde para o Dia 2)...')
    files.download(f'{DATA_DIR}/orders_clean.csv')
    files.download(f'{DATA_DIR}/items_clean.csv')
    files.download(f'{DATA_DIR}/reviews_clean.csv')

### Checkpoint - Bloco 4 (Final do Dia 1)

In [ ]:
# CHECKPOINT 4 - Final do Dia 1
checks = {
    'orders_clean salvo': os.path.exists(f'{DATA_DIR}/orders_clean.csv'),
    'items_clean salvo': os.path.exists(f'{DATA_DIR}/items_clean.csv'),
    'reviews_clean salvo': os.path.exists(f'{DATA_DIR}/reviews_clean.csv'),
}
for check, passed in checks.items():
    status = 'OK' if passed else 'VERIFICAR'
    print(f'  [{status}] {check}')

print('\nVerifique manualmente:')
print('  [ ] Pelo menos 3 perguntas formuladas na seção acima?')
print('  [ ] Cada pergunta tem dados necessários, métrica e impacto preenchidos?')
print('  [ ] Perguntas priorizadas na tabela de impacto x viabilidade?')
print('  [ ] Observações documentadas nos blocos 1 e 3?')
print('  [ ] Formulou pelo menos uma pergunta não-respondível por EDA (abaixo)?')

if all(checks.values()):
    print('\nDia 1 completo! Amanhã você responde suas perguntas.')

### Desafio: Limites da EDA

Formule uma pergunta que você **não consegue** responder apenas com EDA (precisaria de teste estatístico, modelo ou experimento). Explique por quê. Isso exercita a distinção entre o que dados mostram e o que dados provam.

_Sua pergunta não-respondível e justificativa:_

Exemplo: "Reduzir o prazo de entrega em X dias causaria aumento de Y pontos na satisfação?" Não é respondível com EDA porque correlação (atraso associado a nota baixa) não é causalidade. Seria necessário um experimento controlado ou técnicas de inferência causal.

_Sua pergunta:_

_Por que não é respondível apenas com EDA?_
